# 03. Ensemble Species Distribution Modeling

This notebook extends `02_modeling.ipynb` with three additions, following the methodology used in Azrag et al. (2023, Frontiers in Ecology and Evolution) for a closely related fruit pest SDM at the same host institution (icipe):

1. **VIF-based covariate selection**: WorldClim bioclimatic variables are known to be highly intercorrelated. This removes redundant covariates before modeling.
2. **Multi-classifier ensemble**: Random Forest, Gradient Boosting, and Logistic Regression are each trained and cross-validated, then combined into a weighted ensemble (weighted by each classifier's own cross-validated AUC).
3. **ROC curves**: out-of-fold ROC curves per species, alongside the AUC scores already reported in notebook 02.

This notebook loads notebook 02's saved outputs directly (`clean_data.csv`, `occurrence_covariates.csv`, `host_plant_lookup.json`) rather than re-extracting anything from the raw environmental rasters.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, auc

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## Part 1: Load Notebook 02 Outputs

In [ ]:
df = pd.read_csv('../data/clean_data.csv')
covariates = pd.read_csv('../data/occurrence_covariates.csv')

with open('../outputs/models/host_plant_lookup.json') as f:
    host_plant_lookup = json.load(f)

covariate_cols = [
    'bio_1','bio_2','bio_3','bio_4','bio_5','bio_6','bio_7','bio_8','bio_9','bio_10',
    'bio_11','bio_12','bio_13','bio_14','bio_15','bio_16','bio_17','bio_18','bio_19',
    'elevation','clay','ph',
]

print('Clean data:', df.shape)
print('Occurrence covariates:', covariates.shape)
print('Host plant lookup entries:', len(host_plant_lookup))

In [ ]:
species_covariates = covariates[covariates['taxon_resolution'] == 'species']
TARGET_SPECIES = species_covariates['scientific_name'].value_counts().head(5).index.tolist()
print('Modeling these species:', TARGET_SPECIES)

## Part 2: VIF-Based Covariate Selection

In [ ]:
def select_covariates_by_vif(data, cols, threshold=10.0):
    """
    Iteratively drops the covariate with the highest variance inflation factor
    (VIF) until all remaining covariates are below the threshold. WorldClim
    bioclimatic variables are known to be highly intercorrelated (e.g. bio_1,
    bio_10, and bio_11 are all summaries of temperature), which can destabilize
    model coefficients and obscure which variables genuinely drive predictions.
    """
    selected = list(cols)
    while True:
        X = data[selected].dropna()
        vifs = pd.Series(
            [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
            index=selected
        )
        max_vif = vifs.max()
        if max_vif <= threshold or len(selected) <= 2:
            break
        drop_col = vifs.idxmax()
        print(f'Dropping {drop_col} (VIF={max_vif:.1f})')
        selected.remove(drop_col)
    return selected, vifs

covariate_cols_selected, final_vifs = select_covariates_by_vif(covariates, covariate_cols)
print()
print('Selected covariates after VIF reduction:', covariate_cols_selected)
print()
print('Final VIF values:')
print(final_vifs.sort_values(ascending=False))

## Part 3: Target-Group Background and Spatial Blocks

In [ ]:
def get_target_group_background(target_species, all_covariates):
    """Background = all occurrence records of OTHER species in this dataset."""
    return all_covariates[all_covariates['scientific_name'] != target_species].copy()

def add_spatial_block(data, block_size_deg=2.0):
    """Assign each point to a coarse spatial grid cell, used to group points for
    spatial cross-validation so nearby points don't leak between train and test."""
    block_lat = (data['latitude'] // block_size_deg).astype(int)
    block_lon = (data['longitude'] // block_size_deg).astype(int)
    return block_lat.astype(str) + '_' + block_lon.astype(str)

## Part 4: Multi-Classifier Ensemble

Three classifiers are trained per species and combined into a weighted ensemble. Each is weighted by its own spatial cross-validated AUC, so better-performing classifiers contribute more to the final prediction.

In [ ]:
def build_classifiers():
    return {
        'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced'),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
        'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)),
    }

In [ ]:
ensemble_models = {}
results = []

for species in TARGET_SPECIES:
    presence = species_covariates[species_covariates['scientific_name'] == species].copy()
    bg = get_target_group_background(species, covariates)
    presence['block'] = add_spatial_block(presence)
    bg['block'] = add_spatial_block(bg)

    X = pd.concat([presence[covariate_cols_selected], bg[covariate_cols_selected]], ignore_index=True)
    y = np.concatenate([np.ones(len(presence)), np.zeros(len(bg))])
    groups = pd.concat([presence['block'], bg['block']], ignore_index=True)
    n_splits = min(5, groups.nunique())

    classifiers = build_classifiers()
    clf_aucs = {}
    fitted_clfs = {}

    if n_splits >= 2:
        gkf = GroupKFold(n_splits=n_splits)
        for name, clf in classifiers.items():
            scores = cross_val_score(clf, X, y, cv=gkf, groups=groups, scoring='roc_auc')
            clf_aucs[name] = scores.mean()
            clf.fit(X, y)
            fitted_clfs[name] = clf
    else:
        for name, clf in classifiers.items():
            clf_aucs[name] = np.nan
            clf.fit(X, y)
            fitted_clfs[name] = clf

    weights = pd.Series(clf_aucs)
    weights = weights / weights.sum()

    ensemble_models[species] = {'classifiers': fitted_clfs, 'weights': weights.to_dict()}

    row = {'species': species, 'n_presence': len(presence), **clf_aucs}
    results.append(row)

results_df = pd.DataFrame(results)
results_df

In [ ]:
def predict_ensemble(species, X_new, ensemble_models):
    """Weighted average of each classifier's predicted probability, weighted
    by that classifier's own cross-validated AUC."""
    entry = ensemble_models[species]
    preds = np.zeros(len(X_new))
    for name, clf in entry['classifiers'].items():
        preds += entry['weights'][name] * clf.predict_proba(X_new)[:, 1]
    return preds

### Out-of-fold ensemble AUC

The per-classifier AUCs above are each computed independently. This computes the actual out-of-fold AUC of the *combined weighted ensemble*, which is the number that should be reported and compared against the individual classifiers.

In [ ]:
def compute_ensemble_auc(species, presence, background, covariate_cols):
    presence = presence.copy()
    background = background.copy()
    presence['block'] = add_spatial_block(presence)
    background['block'] = add_spatial_block(background)

    X = pd.concat([presence[covariate_cols], background[covariate_cols]], ignore_index=True)
    y = np.concatenate([np.ones(len(presence)), np.zeros(len(background))])
    groups = pd.concat([presence['block'], background['block']], ignore_index=True)
    n_splits = min(5, groups.nunique())
    if n_splits < 2:
        return None, None, None

    gkf = GroupKFold(n_splits=n_splits)
    y_true_all, y_score_all = [], []

    for train_idx, test_idx in gkf.split(X, y, groups):
        classifiers = build_classifiers()
        fold_aucs = {}
        fitted = {}
        for name, clf in classifiers.items():
            clf.fit(X.iloc[train_idx], y[train_idx])
            fold_aucs[name] = roc_auc_score(y[test_idx], clf.predict_proba(X.iloc[test_idx])[:, 1])
            fitted[name] = clf
        w = pd.Series(fold_aucs)
        w = w / w.sum()
        fold_preds = np.zeros(len(test_idx))
        for name, clf in fitted.items():
            fold_preds += w[name] * clf.predict_proba(X.iloc[test_idx])[:, 1]
        y_true_all.extend(y[test_idx])
        y_score_all.extend(fold_preds)

    fpr, tpr, _ = roc_curve(y_true_all, y_score_all)
    return fpr, tpr, auc(fpr, tpr)


ensemble_results = []
roc_data_by_species = {}

for species in TARGET_SPECIES:
    presence = species_covariates[species_covariates['scientific_name'] == species]
    bg = get_target_group_background(species, covariates)
    fpr, tpr, ens_auc = compute_ensemble_auc(species, presence, bg, covariate_cols_selected)
    roc_data_by_species[species] = (fpr, tpr, ens_auc)
    ensemble_results.append({'species': species, 'ensemble_auc': ens_auc})

ensemble_results_df = pd.DataFrame(ensemble_results)
results_df = results_df.merge(ensemble_results_df, on='species')
results_df

## Part 5: Classifier Comparison and ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.2
classifier_names = ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'ensemble_auc']
colors = plt.cm.tab10(np.linspace(0, 1, len(classifier_names)))

for i, name in enumerate(classifier_names):
    ax.bar(x + i * width, results_df[name], width, label=name.replace('_auc', ' (ensemble)'), color=colors[i])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['species'], rotation=20, ha='right')
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.6)
ax.set_ylabel('Spatial cross-validated AUC')
ax.set_title('Classifier comparison and ensemble performance by species', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/classifier_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, species in enumerate(TARGET_SPECIES):
    fpr, tpr, ens_auc = roc_data_by_species[species]
    ax = axes[i]
    if fpr is None:
        ax.set_visible(False)
        continue
    ax.plot(fpr, tpr, color='darkred', linewidth=2, label=f'Ensemble AUC = {ens_auc:.3f}')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    ax.set_xlabel('1 - Specificity (false positive rate)')
    ax.set_ylabel('Sensitivity (true positive rate)')
    ax.set_title(species, fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Ensemble ROC curves by species (spatial out-of-fold predictions)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/roc_curves_by_species.png', dpi=150)
plt.show()

## Part 6: Feature Importance (Random Forest Component)

In [ ]:
importance_rows = []
for species in TARGET_SPECIES:
    rf = ensemble_models[species]['classifiers']['Random Forest']
    importances = pd.Series(rf.feature_importances_, index=covariate_cols_selected)
    top5 = importances.sort_values(ascending=False).head(5)
    for cov, val in top5.items():
        importance_rows.append({'species': species, 'covariate': cov, 'importance': val})

importance_df = pd.DataFrame(importance_rows)
importance_df.pivot(index='species', columns='covariate', values='importance').fillna(0)

## Part 7: Africa Suitability Mapping (Ensemble Version)

**Update `ENV_DIR` below to your environmental layers folder before running.**

In [ ]:
import rasterio, os

ENV_DIR = 'C:/Users/skimanzi/Desktop/Dacini_SMD_project/Environmental_Layers'

env_files = {
    'bio_1': 'wc2.1_10m_bio_1.tif', 'bio_2': 'wc2.1_10m_bio_2.tif',
    'bio_3': 'wc2.1_10m_bio_3.tif', 'bio_4': 'wc2.1_10m_bio_4.tif',
    'bio_5': 'wc2.1_10m_bio_5.tif', 'bio_6': 'wc2.1_10m_bio_6.tif',
    'bio_7': 'wc2.1_10m_bio_7.tif', 'bio_8': 'wc2.1_10m_bio_8.tif',
    'bio_9': 'wc2.1_10m_bio_9.tif', 'bio_10': 'wc2.1_10m_bio_10.tif',
    'bio_11': 'wc2.1_10m_bio_11.tif', 'bio_12': 'wc2.1_10m_bio_12.tif',
    'bio_13': 'wc2.1_10m_bio_13.tif', 'bio_14': 'wc2.1_10m_bio_14.tif',
    'bio_15': 'wc2.1_10m_bio_15.tif', 'bio_16': 'wc2.1_10m_bio_16.tif',
    'bio_17': 'wc2.1_10m_bio_17.tif', 'bio_18': 'wc2.1_10m_bio_18.tif',
    'bio_19': 'wc2.1_10m_bio_19.tif',
    'elevation': 'wc2.1_10m_elev.tif',
    'clay': 'clay_0-5cm_mean_30s.tif',
    'ph': 'phh2o_0-5cm_mean_30s.tif',
}
env_files = {k: os.path.join(ENV_DIR, v) for k, v in env_files.items()}

def _nearest_valid_pixel(band, row, col, max_radius_px):
    for radius in range(0, max_radius_px + 1):
        r0, r1 = max(0, row - radius), min(band.shape[0], row + radius + 1)
        c0, c1 = max(0, col - radius), min(band.shape[1], col + radius + 1)
        window = band[r0:r1, c0:c1]
        if np.any(~np.isnan(window)):
            valid_idx = np.argwhere(~np.isnan(window))
            center = np.array([row - r0, col - c0])
            dists = np.linalg.norm(valid_idx - center, axis=1)
            best = valid_idx[np.argmin(dists)]
            return float(window[best[0], best[1]]), radius
    return np.nan, -1

def extract_covariates(points_df, lat_col='latitude', lon_col='longitude', fill_gaps=False, max_radius_px=15):
    coords = list(zip(points_df[lon_col], points_df[lat_col]))
    result = points_df.copy()
    for name, path in env_files.items():
        with rasterio.open(path) as src:
            values = [v[0] for v in src.sample(coords)]
            values = [np.nan if (isinstance(v, float) and np.isnan(v)) else v for v in values]
            if fill_gaps and any(np.isnan(v) for v in values):
                band = src.read(1)
                for i, (lon, lat) in enumerate(coords):
                    if np.isnan(values[i]):
                        row, col = src.index(lon, lat)
                        filled, _ = _nearest_valid_pixel(band, row, col, max_radius_px)
                        values[i] = filled
            result[name] = values
    return result

In [ ]:
def build_grid(lat_range, lon_range, step=0.5):
    lats = np.arange(lat_range[0], lat_range[1], step)
    lons = np.arange(lon_range[0], lon_range[1], step)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return pd.DataFrame({'latitude': lat_grid.ravel(), 'longitude': lon_grid.ravel()})

AFRICA_LAT_RANGE = (-35, 38)
AFRICA_LON_RANGE = (-18, 52)

africa_grid = build_grid(AFRICA_LAT_RANGE, AFRICA_LON_RANGE, step=0.5)
africa_grid = extract_covariates(africa_grid)
africa_grid = africa_grid.dropna(subset=covariate_cols_selected)
if africa_grid['ph'].max() > 14:
    africa_grid['ph'] = africa_grid['ph'] / 10
print('Africa grid points:', len(africa_grid))

In [ ]:
def flag_novel_conditions(new_data, train_presence, covariate_cols, novelty_threshold=0.15):
    """Flags a location as climatically novel only if more than novelty_threshold
    fraction of covariates fall outside this species' own observed range."""
    novel_flags = pd.DataFrame(index=new_data.index)
    for col in covariate_cols:
        lo, hi = train_presence[col].min(), train_presence[col].max()
        novel_flags[col] = (new_data[col] < lo) | (new_data[col] > hi)
    result = new_data.copy()
    result['n_novel_covariates'] = novel_flags.sum(axis=1)
    result['is_novel'] = (result['n_novel_covariates'] / len(covariate_cols)) > novelty_threshold
    return result

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

def plot_suitability_map(grid_df, title, lon_range, lat_range, presence_points=None, grid_res=300, smooth_sigma=3):
    grid_lon, grid_lat = np.meshgrid(
        np.linspace(lon_range[0], lon_range[1], grid_res),
        np.linspace(lat_range[0], lat_range[1], grid_res)
    )
    points = (grid_df['longitude'], grid_df['latitude'])
    grid_linear = griddata(points, grid_df['suitability'], (grid_lon, grid_lat), method='linear')
    grid_nearest = griddata(points, grid_df['suitability'], (grid_lon, grid_lat), method='nearest')
    grid_z = np.where(np.isnan(grid_linear), grid_nearest, grid_linear)
    grid_z_smooth = gaussian_filter(grid_z, sigma=smooth_sigma)

    fig = plt.figure(figsize=(10, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#e8e8e8', zorder=0)
    mesh = ax.contourf(grid_lon, grid_lat, grid_z_smooth, levels=15, cmap='YlOrRd',
                        alpha=0.85, transform=ccrs.PlateCarree(), zorder=1)
    ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='-', edgecolor='gray', zorder=3)

    if presence_points is not None and len(presence_points) > 0:
        ax.scatter(presence_points['longitude'], presence_points['latitude'],
                   s=25, facecolor='none', edgecolor='blue', linewidth=1.2,
                   transform=ccrs.PlateCarree(), zorder=5, label='Observed occurrences')
        ax.legend(loc='lower left', fontsize=9, framealpha=0.9)

    cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', shrink=0.6, pad=0.05)
    cbar.set_label('Predicted climatic suitability (ensemble)', fontsize=11)

    ax.annotate('N', xy=(0.95, 0.92), xycoords='axes fraction', fontsize=14, fontweight='bold', ha='center')
    ax.annotate('', xy=(0.95, 0.90), xytext=(0.95, 0.82), xycoords='axes fraction',
                arrowprops=dict(facecolor='black', width=3, headwidth=10))

    center_lat = (lat_range[0] + lat_range[1]) / 2
    bar_km = 1000
    start_lon = lon_range[0] + (lon_range[1] - lon_range[0]) * 0.05
    end_lon = start_lon + (bar_km / 111.32 / np.cos(np.radians(center_lat)))
    bar_lat = lat_range[0] + (lat_range[1] - lat_range[0]) * 0.05
    ax.plot([start_lon, end_lon], [bar_lat, bar_lat], color='black', linewidth=3, transform=ccrs.PlateCarree(), zorder=4)
    ax.text((start_lon + end_lon) / 2, bar_lat + 1, f'{bar_km} km', ha='center', fontsize=9, transform=ccrs.PlateCarree())

    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    return fig, ax

In [ ]:
for species in TARGET_SPECIES:
    presence_for_species = species_covariates[species_covariates['scientific_name'] == species]

    grid_checked = flag_novel_conditions(africa_grid, presence_for_species, covariate_cols_selected)
    grid_checked['suitability'] = predict_ensemble(species, grid_checked[covariate_cols_selected], ensemble_models)

    occurrences_africa = presence_for_species[
        (presence_for_species['longitude'] >= AFRICA_LON_RANGE[0]) &
        (presence_for_species['longitude'] <= AFRICA_LON_RANGE[1]) &
        (presence_for_species['latitude'] >= AFRICA_LAT_RANGE[0]) &
        (presence_for_species['latitude'] <= AFRICA_LAT_RANGE[1])
    ]

    fig, ax = plot_suitability_map(
        grid_checked,
        title=f'{species}: ensemble predicted climatic suitability across Africa',
        lon_range=AFRICA_LON_RANGE, lat_range=AFRICA_LAT_RANGE,
        presence_points=occurrences_africa
    )
    safe_name = species.replace(' ', '_')
    plt.savefig(f'../outputs/figures/africa_suitability_ensemble_{safe_name}.png', dpi=200, bbox_inches='tight')
    plt.show()

    novel_pct = grid_checked['is_novel'].mean() * 100
    print(f"{species}: {novel_pct:.1f}% of Africa grid cells are outside this species' own known climate range\n")

## Part 8: Farmer-Facing Risk Lookup (Ensemble Version)

In [ ]:
def suitability_band(score):
    if score >= 0.6:
        return 'HIGH'
    elif score >= 0.4:
        return 'MODERATE'
    else:
        return 'LOW'

def assess_risk_ensemble(lat, lon, crop, ensemble_models, host_plant_lookup, covariate_cols):
    point = pd.DataFrame({'latitude': [lat], 'longitude': [lon]})
    point = extract_covariates(point, fill_gaps=True)

    if point[covariate_cols].isna().any(axis=1).iloc[0]:
        return {'error': 'No environmental data available at this location'}

    if point['ph'].iloc[0] > 14:
        point['ph'] = point['ph'] / 10

    findings = []
    for species in ensemble_models:
        suitability = predict_ensemble(species, point[covariate_cols], ensemble_models)[0]
        hosts = host_plant_lookup.get(species, [])
        is_host = any(crop.lower() in h.lower() or h.lower() in crop.lower() for h in hosts)
        findings.append({
            'species': species,
            'climatic_suitability': round(float(suitability), 3),
            'band': suitability_band(suitability),
            'documented_hosts': hosts,
            'crop_is_documented_host': is_host,
        })

    findings = sorted(findings, key=lambda r: r['climatic_suitability'], reverse=True)
    return {'location': (lat, lon), 'crop_queried': crop, 'species_assessment': findings}

In [ ]:
result = assess_risk_ensemble(
    lat=-1.286389, lon=36.817223, crop='Mangifera indica',
    ensemble_models=ensemble_models, host_plant_lookup=host_plant_lookup,
    covariate_cols=covariate_cols_selected
)

for row in result['species_assessment']:
    host_note = 'documented host' if row['crop_is_documented_host'] else 'not a documented host'
    print(f"{row['species']}: suitability={row['climatic_suitability']} ({row['band']}), {host_note}")

## Save ensemble models

In [ ]:
for species, entry in ensemble_models.items():
    safe_name = species.replace(' ', '_')
    joblib.dump(entry, f'../outputs/models/ensemble_{safe_name}.joblib')

results_df.to_csv('../outputs/models/ensemble_performance_summary.csv', index=False)
importance_df.to_csv('../outputs/models/ensemble_feature_importance.csv', index=False)
print('Ensemble models and performance summary saved to outputs/models/')

## Summary

This notebook extended notebook 02 with VIF-based covariate reduction and a three-classifier
ensemble.

**VIF reduction**: 22 covariates reduced to 8 (bio_4, bio_9, bio_13, bio_14, bio_18, bio_19,
elevation, clay), removing highly intercorrelated temperature and precipitation summaries.

**Classifier comparison**: Random Forest consistently outperformed Gradient Boosting and
Logistic Regression across all five species, matching the same finding in Azrag et al. (2023).
The weighted ensemble matched or approached Random Forest's performance in most cases,
though for *B. tryoni* the ensemble (0.905) slightly underperformed Random Forest alone
(0.919), pulled down by the weaker Logistic Regression component even after weighting.

**Cross-method robustness**: ensemble AUC closely matched the single-Random-Forest results
from notebook 02 despite a completely different covariate set (8 vs 22 variables):

| Species | Notebook 2 (RF, 22 covariates) | Notebook 3 (Ensemble, 8 covariates) |
|---|---|---|
| B. oleae | 0.995 | 0.990 |
| B. tryoni | 0.931 | 0.905 |
| Z. tau | 0.821 | 0.819 |
| B. dorsalis | 0.744 | 0.738 |
| B. cucurbitae | 0.710 | 0.700 |

**Key ecological finding**: precipitation of the wettest month (bio_13) was a top-2 predictor
for four of five species, independently matching the top finding in Azrag et al. (2023) for a
different Tephritidae-adjacent pest, strengthening confidence that this reflects a genuine
regional climate driver rather than a dataset-specific artifact.

**Africa suitability maps**: reproduced the same patterns found in notebook 02, *B. oleae*
restricted to the Mediterranean fringe with near-zero sub-Saharan suitability, *B. dorsalis*
and *B. cucurbitae* showing broad, plausible tropical suitability aligned with real occurrence
points, and *Zeugodacus tau* remaining patchy and low-confidence, consistent with it having
zero African ground truth in either modeling approach.

**Conclusion**: agreement between two independently built pipelines (different covariates,
different classifiers) on the same ecological patterns supports the robustness of the core
findings rather than either result being an artifact of one particular modeling choice.